In [ ]:
import spacy
import numpy as np
import pandas as pd
import re
import collections
import itertools
from sentence_transformers import SentenceTransformer
import hdbscan
import epitran
from scipy import stats
import warnings
warnings.filterwarnings('ignore')


nlp = spacy.load("ru_core_news_sm")
epi = epitran.Epitran('rus-Cyrl')
sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')


corpus = {
    "Туманность Андромеды": "Дар Ветер смотрел на Туманность Андромеды, известную также как М31 или НН89105-СБ23. Звездолет Тантра готовился к старту.",
    "Час Быка": "Чойо Чагас и Шот Шенк обсуждали ситуацию на Тормансе. Земляне Родис и Гентло Ши готовились к высадке на остров Забвения.",
    "Сердце Змеи": "Веда Конг и Мвен Мас отправились к системе Эпсилон Тукана. На Тормансе правили кжи и джи, в то время как Земля входила в Великое Кольцо."
}

def extract_onyms_spacy(text):

    doc = nlp(text)
    candidates = []
    for sent in doc.sents:
        for token in sent:
            # spaCy 3.x совместимый фильтр
            is_proper = token.pos_ == "PROPN"
            is_title = token.text[0].isupper() and len(token.text) > 3 and token.is_alpha
            if is_proper or is_title:
                candidates.append({
                    "text": token.text,
                    "start": token.idx,
                    "end": token.idx + len(token.text)
                })
    return candidates  #  Явный возврат

def assign_context(onym, doc, window=3):

    positions = [m.start() for m in re.finditer(re.escape(onym), doc.text)]
    context_votes = collections.Counter()
    sents_list = list(doc.sents)  # Преобразуем итератор в список

    for pos in positions:
        sent_idx = None
        for i, s in enumerate(sents_list):
            if s.start_char <= pos < s.end_char:
                sent_idx = i
                break
        if sent_idx is None: continue

        start_i = max(0, sent_idx - window)
        end_i = min(len(sents_list), sent_idx + window + 1)
        win_text = " ".join(s.text.lower() for s in sents_list[start_i:end_i])

        for ctx, markers in CONTEXT_MARKERS.items():
            if any(m.lower() in win_text for m in markers):
                context_votes[ctx] += 1

    return context_votes.most_common(1)[0][0] if context_votes else "Неопределено"

def run_full_pipeline(corpus):
    context_df = []
    for novel, text in corpus.items():
        if not text.strip():
            print(f"⚠️ Текст '{novel}' пуст. Пропуск.")
            continue

        doc = nlp(text)
        onyms_raw = extract_onyms_spacy(text)

        # 🛡 Защита от None/пустого списка
        if not onyms_raw:
            print(f"'{novel}' не найдены.")
            continue

        unique_onyms = list(set(o["text"] for o in onyms_raw))
        mapping, clusters = resolve_entities(unique_onyms, min_cluster=2)

        for onym in unique_onyms:
            ctx = assign_context(onym, doc)
            phon_ratio = get_phoneme_ratio(onym)
            canonical = mapping.get(onym, onym)
            cluster_id = next((k for k, v in clusters.items() if onym in v), "noise")

            context_df.append({
                "novel": novel,
                "variant": onym,
                "canonical": canonical,
                "context": ctx,
                "dissonance_score": phon_ratio,
                "cluster_id": cluster_id
            })

    return pd.DataFrame(context_df)


df_result = run_full_pipeline(corpus)
if not df_result.empty:
    print(df_result.head(10))
    print(f"\nВсего  онимов: {len(df_result)}")
else:
    print("\nПайплайн не вернул данных. Проверьте загрузку текстов в переменную `corpus`.")


CONTEXT_MARKERS = {
    "Торманс": ["Торманс", "Чойо Чагас", "Шот Шенк", "Гентло Ши", "Зетрино", "кжи", "джи", "Хонтээло"],
    "Земля":   ["Земля", "Великое Кольцо", "Совет Экономики", "Москва", "Сибирь", "Каспий", "Дар Ветер", "Веда Конг", "Мвен Мас"],
    "Космос":  ["звездолет", "планета", "система", "космос", "андроид", "орбита", "Тантра", "Эпсилон", "Млечный путь"]
}

def assign_context(onym, doc, window=3):

    positions = [m.start() for m in re.finditer(re.escape(onym), doc.text)]
    context_votes = collections.Counter()

    for pos in positions:
        sent_idx = doc.char_span(pos, pos+1)[0].i if doc.char_span(pos, pos+1) else 0
        start = max(0, sent_idx - window)
        end = min(len(list(doc.sents)), sent_idx + window + 1)
        window_text = " ".join([s.text for s in itertools.islice(doc.sents, start, end)]).lower()

        for ctx, markers in CONTEXT_MARKERS.items():
            for m in markers:
                if m.lower() in window_text:
                    context_votes[ctx] += 1

    return context_votes.most_common(1)[0][0] if context_votes else "Неопределено"

#
def resolve_entities(onym_list, min_cluster=2, threshold=0.75):

    unique = list(set(onym_list))
    if len(unique) < min_cluster:
        return {u: [u] for u in unique}

    embeddings = sbert.encode(unique, normalize_embeddings=True)


    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster,
        metric='cosine',
        algorithm='brute',  # <-- Ключевое исправление
        min_samples=1,
        cluster_selection_epsilon=1-threshold
    )
    labels = clusterer.fit_predict(embeddings)

    clusters = collections.defaultdict(list)
    for name, lbl in zip(unique, labels):
        key = f"cluster_{lbl}" if lbl != -1 else "noise"
        clusters[key].append(name)

    mapping = {}
    for key, group in clusters.items():
        canonical = group[0]
        for var in group:
            mapping[var] = canonical
    return mapping, clusters
DISSONANT_PHONEMES = {'tʃ', 't͡ʃ', 'ɡ', 'ʃ', 'z', 'k', 'b', 'x'} # [ч], [г], [ш], [з], [к], [б], [х]

def get_phoneme_ratio(name):

    try:
        ipa = epi.transliterate(name)
        ipa = ipa.replace('ˈ', '').replace('ˌ', '') # Убираем знаки ударения
        phonemes = re.findall(r'[a-zʃt͡ʃʒdʒɡŋɬɮχʁʕʔːˑʰʷʲʲ]', ipa) # Базовый набор
        if not phonemes: return 0.0
        dissonant = sum(1 for p in phonemes if p in DISSONANT_PHONEMES)
        return dissonant / len(phonemes)
    except:
        return np.nan


def run_full_pipeline(corpus):
    all_onyms = []
    context_df = []

    for novel, text in corpus.items():
        doc = nlp(text)
        onyms_raw = extract_onyms_spacy(text)
        unique_onyms = list(set(o["text"] for o in onyms_raw))

        # Entity Resolution
        mapping, clusters = resolve_entities(unique_onyms, min_cluster=2)

        for onym in unique_onyms:
            ctx = assign_context(onym, doc)
            phon_ratio = get_phoneme_ratio(onym)
            canonical = mapping.get(onym, onym)

            context_df.append({
                "novel": novel,
                "variant": onym,
                "canonical": canonical,
                "context": ctx,
                "dissonance_score": phon_ratio,
                "cluster_id": clusters.get(f"cluster_{list(clusters.keys())[0]}", "noise")
            })
            all_onyms.append(onym)

    df = pd.DataFrame(context_df)
    return df


df_result = run_full_pipeline(corpus)
print(df_result.to_string())


df_tormans = df_result[df_result["context"] == "Торманс"]["dissonance_score"].dropna()
df_earth   = df_result[df_result["context"] == "Земля"]["dissonance_score"].dropna()

if len(df_tormans) > 2 and len(df_earth) > 2:
    stat, p_value = stats.mannwhitneyu(df_tormans, df_earth, alternative='greater')
    print(f"\n🔬 Фонетический тест (Торманс vs Земля):")
    print(f"   U={stat:.2f}, p={p_value:.4f}")
    if p_value < 0.05:
        print("  Гипотеза подтверждена: имена Торманса статистически значимо более дисфоничны.")
    else:
        print("   На текущем отрывке гипотеза не подтверждена (требуется полный корпус).")

In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "1Srm4I_dekUG",
    "outputId": "b74137de-192f-408f-e41d-a5f5322f4eb2"
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Requirement already satisfied: pymorphy2 in /usr/local/lib/python3.12/dist-packages (0.9.1)\n",
      "Requirement already satisfied: pandas in /usr/local/lib/python3.12/dist-packages (2.2.2)\n",
      "Requirement already satisfied: numpy in /usr/local/lib/python3.12/dist-packages (2.0.2)\n",
      "Requirement already satisfied: scikit-learn in /usr/local/lib/python3.12/dist-packages (1.6.1)\n",
      "Requirement already satisfied: scipy in /usr/local/lib/python3.12/dist-packages (1.16.3)\n",
      "Requirement already satisfied: seaborn in /usr/local/lib/python3.12/dist-packages (0.13.2)\n",
      "Requirement already satisfied: matplotlib in /usr/local/lib/python3.12/dist-packages (3.10.0)\n",
      "Requirement already satisfied: plotly in /usr/local/lib/python3.12/dist-packages (5.24.1)\n"
     ]
    }
   ],
   "source": "!pip install pymorphy2 pandas numpy scikit-learn scipy seaborn matplotlib plotly"
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "id": "T9coqrdJgFOU",
    "outputId": "a3c24bf9-6163-4949-e462-6ac490d33646"
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m53.9/53.9 kB\u001b[0m \u001b[31m1.9 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m8.4/8.4 MB\u001b[0m \u001b[31m50.5 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[?25h"
     ]
    }
   ],
   "source": "!pip install -q pymorphy3"
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/",
     "height": 1000
    },
    "id": "rDVRwvnQcGac",
    "outputId": "dd454e48-5ea3-4f61-804b-83549db6ebe4"
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "✅ туманность_андромеды.txt прочитан в кодировке utf-8\n",
      "✅ час_быка.txt прочитан в кодировке windows-1251\n",
      "✅ сердце_змеи.txt прочитан в кодировке windows-1251\n",
      "🔝 Характерные онимы по TF-IDF:\n",
      "\n",
      "Туманность Андромеды:\n",
      "       Туманность Андромеды\n",
      "дар                0.507288\n",
      "веда               0.358745\n",
      "мвен               0.322310\n",
      "ветер              0.311202\n",
      "мас                0.306895\n",
      "\n",
      "Час Быка:\n",
      "          Час Быка\n",
      "родис     0.597994\n",
      "фай       0.448496\n",
      "чеди      0.393871\n",
      "гриф      0.206998\n",
      "торманса  0.195498\n",
      "\n",
      "Сердце Змеи:\n",
      "      Сердце Змеи\n",
      "кари     0.507896\n",
      "мут      0.426633\n",
      "анг      0.386001\n",
      "рам      0.284422\n",
      "тэй      0.264106\n",
      "\n",
      "📊 Chi-square test (Земля vs Торманс по романам): χ²=1.905, p=0.3858\n"
     ]
    }
   ],
   "source": [
    "import os\n",
    "import re\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import collections\n",
    "import pymorphy3 as pm3\n",
    "from sklearn.feature_extraction.text import TfidfVectorizer\n",
    "from scipy import stats\n",
    "import seaborn as sns\n",
    "import matplotlib.pyplot as plt\n",
    "import plotly.express as px\n",
    "import plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "import warnings\n",
    "\n",
    "warnings.filterwarnings('ignore')\n",
    "plt.style.use('seaborn-v0_8-whitegrid')\n",
    "\n",
    "files = {\n",
    "    \"Туманность Андромеды\": \"/content/туманность_андромеды.txt\",\n",
    "    \"Час Быка\": \"/content/час_быка.txt\",\n",
    "    \"Сердце Змеи\": \"/content/сердце_змеи.txt\"\n",
    "}\n",
    "\n",
    "def read_cyrillic_file(path):\n",
    "    encodings = ['utf-8', 'windows-1251', 'cp1251', 'koi8-r', 'iso-8859-5']\n",
    "    for enc in encodings:\n",
    "        try:\n",
    "            with open(path, 'r', encoding=enc) as f:\n",
    "                content = f.read()\n",
    "            print(f\"✅ {os.path.basename(path)} прочитан в кодировке {enc}\")\n",
    "            return content\n",
    "        except UnicodeDecodeError:\n",
    "            continue\n",
    "    raise ValueError(f\"❌ Не удалось прочитать {path}. Проверьте кодировку файла.\")\n",
    "\n",
    "texts = {}\n",
    "for name, path in files.items():\n",
    "    if os.path.exists(path):\n",
    "        texts[name] = read_cyrillic_file(path)\n",
    "    else:\n",
    "        print(f\"⚠️ Файл {path} не найден. Проверьте пути.\")\n",
    "\n",
    "morph = pm3.MorphAnalyzer()\n",
    "STOP_START = {'И','А','Но','О','В','К','С','У','На','По','Для','Как','Так','Что','Если','Когда','Где','Кто','Этот','Тот','Все','Они','Мы','Он','Она','Оно','Они'}\n",
    "\n",
    "def extract_onyms(text):\n",
    "    pattern = r'\\b[А-ЯЁ][а-яё]{2,}(?:[-\\'][А-ЯЁ][а-яё]{1,})*\\b'\n",
    "    candidates = re.findall(pattern, text)\n",
    "\n",
    "    onyms = []\n",
    "    for cand in candidates:\n",
    "        if cand in STOP_START or len(cand) < 3:\n",
    "            continue\n",
    "        parsed = morph.parse(cand.split('-')[0])[0]\n",
    "        if 'Name' in str(parsed.tag) or parsed.tag.POS == 'NOUN' and 'Name' in str(parsed.tag):\n",
    "            onyms.append(cand)\n",
    "        elif cand[0].isupper() and not parsed.tag.POS in {'PREP','CONJ','PART','PRCL'}:\n",
    "            onyms.append(cand)\n",
    "\n",
    "    return onyms\n",
    "\n",
    "corpus_onyms = {novel: extract_onyms(text) for novel, text in texts.items()}\n",
    "\n",
    "TYPE_DICT = {\n",
    "    'Дар Ветер':'антропоним', 'Чара Нанди':'антропоним', 'Веда Конг':'антропоним',\n",
    "    'Мвен Мас':'антропоним', 'Эрг Ноор':'антропоним', 'Родис':'антропоним',\n",
    "    'Чойо Чагас':'антропоним', 'Шот Шенк':'антропоним', 'Гентло Ши':'антропоним',\n",
    "    'Земля':'топоним', 'Торманс':'топоним', 'Эпсилон Тукана':'астроним',\n",
    "    'Туманность Андромеды':'космоним', 'Остров Забвения':'топоним',\n",
    "    'Великое Кольцо':'эргоним', 'Совет Экономики':'эргоним',\n",
    "    'Темное Пламя':'порейоним', 'Тантра':'порейоним', 'Парус':'порейоним',\n",
    "    'Зирда':'астроним', 'К2-2Н-88':'астроним'\n",
    "}\n",
    "\n",
    "REAL_DICT = {'Земля', 'Обь', 'Гренландия', 'Красное море', 'Сибирь', 'Каспий',\n",
    "             'Иртыш', 'Атлантический океан', 'Тритон', 'Ганимед', 'Калисто',\n",
    "             'Млечный путь', 'Нефуд', 'Атакама', 'Намиб'}\n",
    "\n",
    "TORMANS_NAMES = [\n",
    "    'Торманс', 'Чойо Чагас', 'Шот Шенк', 'Гентло Ши', 'Зетрино Умрог',\n",
    "    'Кандо Лелуф', 'Янтре Яхах', 'Хонтээло Толло Фраэль', 'Чадмо Сонте Тазтот'\n",
    "]\n",
    "\n",
    "PLANET_MAP = {\n",
    "    'Торманс': TORMANS_NAMES,\n",
    "    'Земля':  [name for name in TYPE_DICT.keys() if name not in TORMANS_NAMES]\n",
    "}\n",
    "\n",
    "def classify_onym(name):\n",
    "    t = TYPE_DICT.get(name, 'неклассифицировано')\n",
    "    r = 'реальный' if name in REAL_DICT else 'вымышленный'\n",
    "    p = 'Торманс' if name in PLANET_MAP['Торманс'] else ('Земля' if name in TYPE_DICT else 'иное')\n",
    "    return pd.Series({'type': t, 'reality': r, 'planet': p})\n",
    "\n",
    "records = []\n",
    "for novel, onym_list in corpus_onyms.items():\n",
    "    counter = collections.Counter(onym_list)\n",
    "    for onym, freq in counter.items():\n",
    "        meta = classify_onym(onym)\n",
    "        meta['onym'] = onym\n",
    "        meta['freq'] = freq\n",
    "        meta['novel'] = novel\n",
    "        records.append(meta)\n",
    "\n",
    "df = pd.DataFrame(records)\n",
    "\n",
    "pivot_freq = df.pivot_table(index='onym', columns='novel', values='freq', aggfunc='sum', fill_value=0)\n",
    "pivot_type = pd.crosstab(df['type'], df['novel'])\n",
    "pivot_reality = pd.crosstab(df['reality'], df['novel'])\n",
    "\n",
    "docs = [\" \".join(onyms) for onyms in corpus_onyms.values()]\n",
    "tfidf = TfidfVectorizer(token_pattern=r'[^\\s]+')\n",
    "tfidf_matrix = tfidf.fit_transform(docs)\n",
    "tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out(), index=files.keys())\n",
    "tfidf_df = tfidf_df.T.sort_values(by=\"Туманность Андромеды\", ascending=False)\n",
    "\n",
    "print(\"🔝 Характерные онимы по TF-IDF:\")\n",
    "for novel in files.keys():\n",
    "    top = tfidf_df[[novel]].dropna().sort_values(novel, ascending=False).head(5)\n",
    "    print(f\"\\n{novel}:\\n{top.to_string()}\")\n",
    "\n",
    "df_planet = df[df['planet'].isin(['Земля', 'Торманс'])]\n",
    "contingency = pd.crosstab(df_planet['novel'], df_planet['planet'])\n",
    "chi2, p, dof, expected = stats.chi2_contingency(contingency)\n",
    "print(f\"\\n📊 Chi-square test (Земля vs Торманс по романам): χ²={chi2:.3f}, p={p:.4f}\")\n",
    "\n",
    "fig, axes = plt.subplots(2, 2, figsize=(16, 12))\n",
    "\n",
    "top10 = df.groupby('onym')['freq'].sum().sort_values(ascending=False).head(10)\n",
    "sns.barplot(x=top10.values, y=top10.index, ax=axes[0,0], palette='viridis')\n",
    "axes[0,0].set_title('Топ-10 по суммарной частоте')\n",
    "axes[0,0].set_xlabel('Частота')\n",
    "axes[0,0].set_ylabel('Оним')\n",
    "\n",
    "pivot_type.plot(kind='bar', ax=axes[0,1], color=sns.color_palette('Set2'))\n",
    "axes[0,1].set_title('Распределение типов по романам')\n",
    "axes[0,1].legend(title='Тип', bbox_to_anchor=(1.05, 1))\n",
    "\n",
    "reality_counts = df['reality'].value_counts()\n",
    "axes[1,0].pie(reality_counts.values, labels=reality_counts.index,\n",
    "              autopct='%1.1f%%', colors=['#4CAF50', '#F44336'], startangle=90)\n",
    "axes[1,0].set_title('Соотношение реальных/вымышленных')\n",
    "\n",
    "variants = df[df['onym'].str.contains('Андромед|М31|НН89105|Звёздный', case=False, na=False, regex=True)]\n",
    "if not variants.empty:\n",
    "    sns.barplot(x='novel', y='freq', hue='onym', data=variants, ax=axes[1,1], palette='coolwarm')\n",
    "    axes[1,1].set_title('Вариативность номинации (Туманность Андромеды и аналоги)')\n",
    "    axes[1,1].legend(title='Вариант', fontsize=8)\n",
    "else:\n",
    "    axes[1,1].text(0.5, 0.5, 'Варианты не найдены в корпусе',\n",
    "                   ha='center', va='center', transform=axes[1,1].transAxes)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "corr_df = pivot_freq.corr()\n",
    "fig_heat = px.imshow(corr_df, text_auto='.2f', aspect='auto',\n",
    "                     color_continuous_scale='RdBu_r',\n",
    "                     title='Корреляция частот онимов между романами')\n",
    "fig_heat.update_layout(height=500)\n",
    "fig_heat.show()\n",
    "\n",
    "df.to_csv('efremov_onyms_analysis.csv', index=False, encoding='utf-8-sig')\n",
    "print(\"\\n✅ Результаты сохранены в efremov_onyms_analysis.csv\")\n",
    "print(\"📌 Для академической валидации рекомендуется:\")\n",
    "print(\"   1. Дополнить TYPE_DICT/REAL_DICT на основе ручного разметки выборки.\")\n",
    "print(\"   2. Заменить regex-экстракцию на fine-tuned ruBert-NER или Natasha.\")\n",
    "print(\"   3. Использовать контекстные окна (±5 предложений) для прагматического анализа.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "NcKE0WLbjjUY",
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "outputId": "2f00e061-6929-4b93-c03f-8717c6efe89b"
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "  Preparing metadata (setup.py) ... \u001b[?25l\u001b[?25hdone\n",
      "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m34.4/34.4 MB\u001b[0m \u001b[31m36.1 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m221.3/221.3 kB\u001b[0m \u001b[31m15.5 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m78.9/78.9 kB\u001b[0m \u001b[31m6.1 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m46.7/46.7 kB\u001b[0m \u001b[31m3.7 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m1.5/1.5 MB\u001b[0m \u001b[31m73.7 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[?25h  Building wheel for unicodecsv (setup.py) ... \u001b[?25l\u001b[?25hdone\n",
      "Collecting ru-core-news-sm==3.8.0\n",
      "  Downloading https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.8.0/ru_core_news_sm-3.8.0-py3-none-any.whl (15.3 MB)\n",
      "\u001b[2K     \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m15.3/15.3 MB\u001b[0m \u001b[31m88.5 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[?25hRequirement already satisfied: pymorphy3>=1.0.0 in /usr/local/lib/python3.12/dist-packages (from ru-core-news-sm==3.8.0) (2.0.6)\n",
      "Requirement already satisfied: dawg2-python>=0.8.0 in /usr/local/lib/python3.12/dist-packages (from pymorphy3>=1.0.0->ru-core-news-sm==3.8.0) (0.9.0)\n",
      "Requirement already satisfied: pymorphy3-dicts-ru in /usr/local/lib/python3.12/dist-packages (from pymorphy3>=1.0.0->ru-core-news-sm==3.8.0) (2.4.417150.4580142)\n",
      "Requirement already satisfied: setuptools>=68.2.2 in /usr/local/lib/python3.12/dist-packages (from pymorphy3>=1.0.0->ru-core-news-sm==3.8.0) (75.2.0)\n",
      "Installing collected packages: ru-core-news-sm\n",
      "Successfully installed ru-core-news-sm-3.8.0\n",
      "\u001b[38;5;2m✔ Download and installation successful\u001b[0m\n",
      "You can now load the package via spacy.load('ru_core_news_sm')\n",
      "\u001b[38;5;3m⚠ Restart to reload dependencies\u001b[0m\n",
      "If you are in a Jupyter or Colab notebook, you may need to restart Python in\n",
      "order to load all the package's dependencies. You can do this by selecting the\n",
      "'Restart kernel' or 'Restart runtime' option.\n"
     ]
    }
   ],
   "source": [
    "!pip install -q spacy natasha sentence-transformers hdbscan epitran scipy pandas numpy\n",
    "!python -m spacy download ru_core_news_sm"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "DQjbZ2tLjBOo",
    "colab": {
     "base_uri": "https://localhost:8080/",
     "height": 860,
     "referenced_widgets": [
      "0823e4a106b04f0aa05450dc8dd88863",
      "4781ef6049d34fb08539f5b9cc50dc9a",
      "20e51f88cb6e4eb7801644e03d6aed1b",
      "8049e03887804b8ea52cfca15d6455ce",
      "44bb6f3eb41647048638a4c38d466f0b",
      "9ef1bc85bf29431e8b12b9ed236fc4d8",
      "00afe49851a24033ae4f5fa17a41a5b2",
      "7ddd18ff651c4ab9bc2d09b0372df1b6",
      "bdab09514062477d9c8f51a302a1cce6",
      "5b5a3386858d4dc0b62119a2b29bed10",
      "24540c09d9774efa883db61032089757",
      "d48ca6bf00aa44f4be15f15ebc581a17",
      "a7c5755ba165451f8087d54336580881",
      "6e7be52238304c18b3496c5874a190f1",
      "4c6af595f608489b82f6d4aa71526e88",
      "58c15ead0fd945538327730715b800b2",
      "3bf6d6a5f4ca4fdca8f7c45d3402ed61",
      "584c09089a4941ad9a8f268525bc8e92",
      "3f086c91f7cc4dd9a245e557ad12d0ed",
      "b2f78c69776d4469be1a74367419d94d",
      "78df62badf70499982a9e774c46d5f14",
      "106d1f78ff154365a00d08e577ab776f",
      "7c3d1a5cd6c34eeaaf5cabe75d2b1501",
      "de50a06399f1404097c5763d6df2d306",
      "6e688467ec694827a232cda398d8cc4e",
      "56da4589979248448931fa315454a712",
      "b70870b8a16e416c9422105454cba89a",
      "1bf4bf74e717458583482580f30df66a",
      "20d2255894ff4372af326fe87f1fd2c9",
      "1c25d0c799894738b585c72679bb1fa3",
      "becaf493e2b649a7952e826184c3788f",
      "b696da0536694c1aaacf55919064b93c",
      "b7b0617f782f49128e3133c544f04173",
      "ddf465e61199427bafcf21ec0e9ec9bf",
      "88c886051b5c42c2884e56de1da63fff",
      "6798fd4f1c53416a9a869397d90068db",
      "b2915d59d6fd437198316edc05a2f1b4",
      "0fe56502e4e64f5d815be3b856faf8bf",
      "378391402ba14f66bb85fc1f3f7657eb",
      "e059d78c82384eb18c6c66bcb49b639c",
      "3160d09650d540f7937a0f0a585bb535",
      "8a0ec69a3da44f6aa7920aa6991cbd71",
      "358dbe6d293746d190bc1159dadd9f75",
      "c8393775910a4d8397317644c789848b",
      "d63dd75655444f04b7a0c3a1283ab5fb",
      "0ac83af9d05a4f46ba0ac117036d9d6b",
      "3ac7f3d6d31c4ed68e9aa9a809a1e370",
      "a2062f2ff0624e008655e5993b009bf8",
      "5dbab960f6d848f3af85cd401b256b4f",
      "c3440da6c1064153b1aaafda5a275359",
      "a2351daf4d4b4bdc852219d31482f05d",
      "c863e404693e49eb9e82e9ab19b7fb20",
      "0e26eefaf349413195f5aaca46385cf0",
      "13642551144c4ce8a3e0b076eff6a76f",
      "2df832a616914d778875444d36e8128d",
      "9342eee8f89c494d83938717d802553e",
      "e8d5bd2b973f4307a2e5ff4d6412de7f",
      "930e957e904447d9935f386e4f888937",
      "eb5528a5e33e4c0f8da474e2126f946c",
      "6034978d3e1847ed9bbc64d43aa30dfb",
      "28f0710333ba4842b4c2383b298081bf",
      "8b2601cb72a944828f8120b8d1c2d4d8",
      "b78db20a8e024fc6bf0b69ec2a67c9f7",
      "adb58e7f15bf4871a70e944ede3929e8",
      "64ea0c41cf684a7b8cd2c33ccf2d7e51",
      "9216a2aa09574264a6e9cb34240946fa",
      "1f88e99a81634e5d97efd5d0e687d267",
      "28776c4591214cc7b9278a517627fb0f",
      "eb5478c0896d470a8f021df61d28a64d",
      "96cdfa5bd5e84146b97ed5934157745f",
      "72b99f6f0cf94bc4ad5eed7e1bf3e780",
      "3facd370370847f2baef1ff15488834d",
      "8ab89a386a7549f391c1f2d8cd42e5c5",
      "dc9deb7d7e754f0892355530da6bad94",
      "e0dbdc3062b144498b9de40ecae17579",
      "13c29746992948ecb076327c6d9432cd",
      "b06279aa35964a57b09ae8e644177ef9",
      "0665137955c645a8abd78378e055b67e",
      "c6b87d2a369b4be8b7d2cc65779688b9",
      "dd870000360b47588d99fdb55b63e8a4",
      "13d1347769b64f26ae9437ef6fe2a493",
      "a5f4002b2f7b40049d485d3ec8c55531",
      "db4757cf4ce04594b8deed94fdf139de",
      "2239bb05a1e7427cad3ffb12caf4e797",
      "8b74a96d2fb8403997f03c9894aefaf6",
      "7bac8e497bf344fab46b558f1d379526",
      "19c883738ff9491195cf90dcae87dbb8",
      "bffd3302a3c847d5933d93c0f60b086c",
      "4de9b1a9378146e3bf91262bfd9c6690",
      "10515aba512d4d58ba2cf1bc9eef59b3",
      "3d4ae17959a84b92a601b13442d04903",
      "c19526cb156d403794610e3e8fb2d47d",
      "94ce483b83d74ab882b1c2f8d035512f",
      "c080c358b5d74858ac6e765d3d59e6a2",
      "ad409089a2044cf188c004054ae1a766",
      "17c070406f464a98b26905ed82a932b7",
      "f20f5997ab354f67b9bc0f7f30fcedd1",
      "1cac8cc3403345d6b657795216faf8d8",
      "7d48f857d4594ba1ad44b4bb33a8322a",
      "451db23e98c74247ac3ac95c355a7a80",
      "654eded8d6b44f99be3e599a8764ea0a",
      "d08e90a67ed8479aa77186199aeeecc7",
      "37a491ddd224442dbd0f5163d216e720",
      "2c1d31be69674a6cb1d637a8d840b8f8",
      "5595ba6ad292418d8085b0dae0559cd3",
      "998dcd8ab1eb41108bfcf4e43b2fdd7e",
      "6eaf81c364ae4988ad06d4678a58c3d7",
      "11ed2ddd61a24eab9b2051bc17b33f10",
      "40cbd9b980c24f56acbd9b4980439b7e",
      "aa2f0087303c46da80d99821778e7659",
      "0bc6a2acfac24eb4a9da3e0b3e879919",
      "0a83d4e13e7e46f3a9b9f14ce0e85708",
      "f316b1f01e4342e8aba825c3b42442a0",
      "b001dc86da43423885052b8b233035c5",
      "9b115e163a944b22940cfddba4713a98",
      "3573d134733b4222b31e3f637a8b50bf",
      "3276e0e9e4e5481390bcc8a16423fa29",
      "a4af73594c6f402ba17aa8b3d8900ef2",
      "00d1dfe3229c4d69a1a176205c03ca79",
      "7239633bb48b448a977861d003fb77cc",
      "d8ce0a153bff4fe0bc92f30701feb4fc"
     ]
    },
    "outputId": "6c82b25d-dee3-43ba-8578-b04a8baa9d42"
   },
   "execution_count": null,
   "outputs": [
    {
     "output_type": "error",
     "ename": "NameError",
     "evalue": "name 'resolve_entities' is not defined",
     "traceback": [
      "\u001b[0;31m---------------------------------------------------------------------------\u001b[0m",
      "\u001b[0;31mNameError\u001b[0m                                 Traceback (most recent call last)",
      "\u001b[0;32m/tmp/ipykernel_23569/337150957.py\u001b[0m in \u001b[0;36m<cell line: 0>\u001b[0;34m()\u001b[0m",
      "\u001b[1;32m    113\u001b[0m \u001b[0;34m\u001b[0m\u001b[0m",
      "\u001b[1;32m    114\u001b[0m \u001b[0;31m# 🚀 Запуск\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m",
      "\u001b[0;32m--> 115\u001b[0;31m \u001b[0mdf_result\u001b[0m \u001b[0;34m=\u001b[0m \u001b[0mrun_full_pipeline\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0mcorpus\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m",
      "\u001b[0m\u001b[1;32m    116\u001b[0m \u001b[0;32mif\u001b[0m \u001b[0;32mnot\u001b[0m \u001b[0mdf_result\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mempty\u001b[0m\u001b[0;34m:\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m",
      "\u001b[1;32m    117\u001b[0m     \u001b[0mprint\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0mdf_result\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mhead\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0;36m10\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m"
     ]
    }
   ],
   "source": [
    "import spacy\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import re\n",
    "import collections\n",
    "import itertools\n",
    "from sentence_transformers import SentenceTransformer\n",
    "import hdbscan\n",
    "import epitran\n",
    "from scipy import stats\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "nlp = spacy.load(\"ru_core_news_sm\")\n",
    "epi = epitran.Epitran('rus-Cyrl')\n",
    "sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')\n",
    "\n",
    "corpus = {\n",
    "    \"Туманность Андромеды\": \"Дар Ветер смотрел на Туманность Андромеды, известную также как М31 или НН89105-СБ23. Звездолет Тантра готовился к старту.\",\n",
    "    \"Час Быка\": \"Чойо Чагас и Шот Шенк обсуждали ситуацию на Тормансе. Земляне Родис и Гентло Ши готовились к высадке на остров Забвения.\",\n",
    "    \"Сердце Змеи\": \"Веда Конг и Мвен Мас отправились к системе Эпсилон Тукана. На Тормансе правили кжи и джи, в то время как Земля входила в Великое Кольцо.\"\n",
    "}\n",
    "\n",
    "def extract_onyms_spacy(text):\n",
    "    doc = nlp(text)\n",
    "    candidates = []\n",
    "    for sent in doc.sents:\n",
    "        for token in sent:\n",
    "            is_proper = token.pos_ == \"PROPN\"\n",
    "            is_title = token.text[0].isupper() and len(token.text) > 3 and token.is_alpha\n",
    "            if is_proper or is_title:\n",
    "                candidates.append({\n",
    "                    \"text\": token.text,\n",
    "                    \"start\": token.idx,\n",
    "                    \"end\": token.idx + len(token.text)\n",
    "                })\n",
    "    return candidates\n",
    "\n",
    "def assign_context(onym, doc, window=3):\n",
    "    positions = [m.start() for m in re.finditer(re.escape(onym), doc.text)]\n",
    "    context_votes = collections.Counter()\n",
    "    sents_list = list(doc.sents)\n",
    "\n",
    "    for pos in positions:\n",
    "        sent_idx = None\n",
    "        for i, s in enumerate(sents_list):\n",
    "            if s.start_char <= pos < s.end_char:\n",
    "                sent_idx = i\n",
    "                break\n",
    "        if sent_idx is None: continue\n",
    "\n",
    "        start_i = max(0, sent_idx - window)\n",
    "        end_i = min(len(sents_list), sent_idx + window + 1)\n",
    "        win_text = \" \".join(s.text.lower() for s in sents_list[start_i:end_i])\n",
    "\n",
    "        for ctx, markers in CONTEXT_MARKERS.items():\n",
    "            if any(m.lower() in win_text for m in markers):\n",
    "                context_votes[ctx] += 1\n",
    "\n",
    "    return context_votes.most_common(1)[0][0] if context_votes else \"Неопределено\"\n",
    "\n",
    "def resolve_entities(onym_list, min_cluster=2, threshold=0.75):\n",
    "    unique = list(set(onym_list))\n",
    "    if len(unique) < min_cluster:\n",
    "        return {u: [u] for u in unique}\n",
    "\n",
    "    embeddings = sbert.encode(unique, normalize_embeddings=True)\n",
    "    clusterer = hdbscan.HDBSCAN(\n",
    "        min_cluster_size=min_cluster,\n",
    "        metric='cosine',\n",
    "        algorithm='brute',\n",
    "        min_samples=1,\n",
    "        cluster_selection_epsilon=1-threshold\n",
    "    )\n",
    "    labels = clusterer.fit_predict(embeddings)\n",
    "\n",
    "    clusters = collections.defaultdict(list)\n",
    "    for name, lbl in zip(unique, labels):\n",
    "        key = f\"cluster_{lbl}\" if lbl != -1 else \"noise\"\n",
    "        clusters[key].append(name)\n",
    "\n",
    "    mapping = {}\n",
    "    for key, group in clusters.items():\n",
    "        canonical = group[0]\n",
    "        for var in group:\n",
    "            mapping[var] = canonical\n",
    "    return mapping, clusters\n",
    "\n",
    "DISSONANT_PHONEMES = {'tʃ', 't͡ʃ', 'ɡ', 'ʃ', 'z', 'k', 'b', 'x'}\n",
    "\n",
    "def get_phoneme_ratio(name):\n",
    "    try:\n",
    "        ipa = epi.transliterate(name)\n",
    "        ipa = ipa.replace('ˈ', '').replace('ˌ', '')\n",
    "        phonemes = re.findall(r'[a-zʃt͡ʃʒdʒɡŋɬɮχʁʕʔːˑʰʷʲʲ]', ipa)\n",
    "        if not phonemes: return 0.0\n",
    "        dissonant = sum(1 for p in phonemes if p in DISSONANT_PHONEMES)\n",
    "        return dissonant / len(phonemes)\n",
    "    except:\n",
    "        return np.nan\n",
    "\n",
    "def run_full_pipeline(corpus):\n",
    "    all_onyms = []\n",
    "    context_df = []\n",
    "\n",
    "    for novel, text in corpus.items():\n",
    "        if not text.strip():\n",
    "            print(f\"⚠️ Текст '{novel}' пуст. Пропуск.\")\n",
    "            continue\n",
    "\n",
    "        doc = nlp(text)\n",
    "        onyms_raw = extract_onyms_spacy(text)\n",
    "\n",
    "        if not onyms_raw:\n",
    "            print(f\"⚠️ В романе '{novel}' онимы не найдены. Проверьте модель spaCy.\")\n",
    "            continue\n",
    "\n",
    "        unique_onyms = list(set(o[\"text\"] for o in onyms_raw))\n",
    "        mapping, clusters = resolve_entities(unique_onyms, min_cluster=2)\n",
    "\n",
    "        for onym in unique_onyms:\n",
    "            ctx = assign_context(onym, doc)\n",
    "            phon_ratio = get_phoneme_ratio(onym)\n",
    "            canonical = mapping.get(onym, onym)\n",
    "            cluster_id = next((k for k, v in clusters.items() if onym in v), \"noise\")\n",
    "\n",
    "            context_df.append({\n",
    "                \"novel\": novel,\n",
    "                \"variant\": onym,\n",
    "                \"canonical\": canonical,\n",
    "                \"context\": ctx,\n",
    "                \"dissonance_score\": phon_ratio,\n",
    "                \"cluster_id\": cluster_id\n",
    "            })\n",
    "            all_onyms.append(onym)\n",
    "\n",
    "    df = pd.DataFrame(context_df)\n",
    "    return df\n",
    "\n",
    "df_result = run_full_pipeline(corpus)\n",
    "if not df_result.empty:\n",
    "    print(df_result.head(10))\n",
    "    print(f\"\\n✅ Всего обработано онимов: {len(df_result)}\")\n",
    "else:\n",
    "    print(\"\\n❌ Пайплайн не вернул данных. Проверьте загрузку текстов в переменную `corpus`.\")\n",
    "\n",
    "CONTEXT_MARKERS = {\n",
    "    \"Торманс\": [\"Торманс\", \"Чойо Чагас\", \"Шот Шенк\", \"Гентло Ши\", \"Зетрино\", \"кжи\", \"джи\", \"Хонтээло\"],\n",
    "    \"Земля\":   [\"Земля\", \"Великое Кольцо\", \"Совет Экономики\", \"Москва\", \"Сибирь\", \"Каспий\", \"Дар Ветер\", \"Веда Конг\", \"Мвен Мас\"],\n",
    "    \"Космос\":  [\"звездолет\", \"планета\", \"система\", \"космос\", \"андроид\", \"орбита\", \"Тантра\", \"Эпсилон\", \"Млечный путь\"]\n",
    "}\n",
    "\n",
    "df_tormans = df_result[df_result[\"context\"] == \"Торманс\"][\"dissonance_score\"].dropna()\n",
    "df_earth   = df_result[df_result[\"context\"] == \"Земля\"][\"dissonance_score\"].dropna()\n",
    "\n",
    "if len(df_tormans) > 2 and len(df_earth) > 2:\n",
    "    stat, p_value = stats.mannwhitneyu(df_tormans, df_earth, alternative='greater')\n",
    "    print(f\"\\n🔬 Фонетический тест (Торманс vs Земля):\")\n",
    "    print(f\"   U={stat:.2f}, p={p_value:.4f}\")\n",
    "    if p_value < 0.05:\n",
    "        print(\"   ✅ Гипотеза подтверждена: имена Торманса статистически значимо более дисфоничны.\")\n",
    "    else:\n",
    "        print(\"   ⚠️ На текущем отрывке гипотеза не подтверждена (требуется полный корпус).\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "id": "LHzQI5j6qFCL",
    "colab": {
     "base_uri": "https://localhost:8080/"
    },
    "outputId": "1b78f476-ba85-4161-a3e6-d9de4c6ce24a"
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "\u001b[?25l   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m0.0/756.0 kB\u001b[0m \u001b[31m?\u001b[0m eta \u001b[36m-:--:--\u001b[0m\r\u001b[2K   \u001b[91m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[91m╸\u001b[0m\u001b[90m━━\u001b[0m \u001b[32m716.8/756.0 kB\u001b[0m \u001b[31m22.3 MB/s\u001b[0m eta \u001b[36m0:00:01\u001b[0m\r\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m756.0/756.0 kB\u001b[0m \u001b[31m13.9 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[?25h\u001b[?25l   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m0.0/4.9 MB\u001b[0m \u001b[31m?\u001b[0m eta \u001b[36m-:--:--\u001b[0m\r\u001b[2K   \u001b[91m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m\u001b[91m╸\u001b[0m \u001b[32m4.9/4.9 MB\u001b[0m \u001b[31m173.6 MB/s\u001b[0m eta \u001b[36m0:00:01\u001b[0m\r\u001b[2K   \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m4.9/4.9 MB\u001b[0m \u001b[31m95.4 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[?25hCollecting ru-core-news-sm==3.8.0\n",
      "  Downloading https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.8.0/ru_core_news_sm-3.8.0-py3-none-any.whl (15.3 MB)\n",
      "\u001b[2K     \u001b[90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\u001b[0m \u001b[32m15.3/15.3 MB\u001b[0m \u001b[31m54.3 MB/s\u001b[0m eta \u001b[36m0:00:00\u001b[0m\n",
      "\u001b[?25hRequirement already satisfied: pymorphy3>=1.0.0 in /usr/local/lib/python3.12/dist-packages (from ru-core-news-sm==3.8.0) (2.0.6)\n",
      "Requirement already satisfied: dawg2-python>=0.8.0 in /usr/local/lib/python3.12/dist-packages (from pymorphy3>=1.0.0->ru-core-news-sm==3.8.0) (0.9.0)\n",
      "Requirement already satisfied: pymorphy3-dicts-ru in /usr/local/lib/python3.12/dist-packages (from pymorphy3>=1.0.0->ru-core-news-sm==3.8.0) (2.4.417150.4580142)\n",
      "Requirement already satisfied: setuptools>=68.2.2 in /usr/local/lib/python3.12/dist-packages (from pymorphy3>=1.0.0->ru-core-news-sm==3.8.0) (75.2.0)\n",
      "\u001b[38;5;2m✔ Download and installation successful\u001b[0m\n",
      "You can now load the package via spacy.load('ru_core_news_sm')\n",
      "\u001b[38;5;3m⚠ Restart to reload dependencies\u001b[0m\n",
      "If you are in a Jupyter or Colab notebook, you may need to restart Python in\n",
      "order to load all the package's dependencies. You can do this by selecting the\n",
      "'Restart kernel' or 'Restart runtime' option.\n"
     ]
    }
   ],
   "source": [
    "!pip install -q spacy sentence-transformers networkx pyvis pandas numpy\n",
    "!python -m spacy download ru_core_news_sm"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {
    "colab": {
     "base_uri": "https://localhost:8080/",
     "height": 471,
     "referenced_widgets": [
      "a8e662d066ed4839bba0c14ba076d8c7",
      "f7abdccf049946bcb4cf739607dd8bb5",
      "8204d7ab85414f69a3696e725bc21665",
      "e51cc0f248db4699baca482ae2db93be",
      "91c763ade0c8480fa22282d7d2ce834d",
      "ac8573099e314417bc0d3f4cb1950f4e",
      "39cd29cbca234d70b18634c2cf42e1f4",
      "446ad77cda1843599d6d700629c72d68",
      "0a4169c0749a41febad75cf2db6dd982",
      "847bf70b870343b3a2370ea82bd54a37",
      "b82e91b292794ea785731472306c7819"
     ]
    },
    "id": "VxALXTxEqrKL",
    "outputId": "17617cce-afaa-4718-bd13-2f02da83276b"
   },
   "execution_count": null,
   "outputs": [
    {
     "output_type": "stream",
     "name": "stderr",
     "text": [
      "Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n",
      "WARNING:huggingface_hub.utils._http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n"
     ]
    },
    {
     "output_type": "stream",
     "name": "stdout",
     "text": [
      "✅ Туманность Андромеды загружен (кодировка: utf-8)\n",
      "✅ Час Быка загружен (кодировка: windows-1251)\n",
      "✅ Сердце Змеи загружен (кодировка: windows-1251)\n"
     ]
    },
    {
     "output_type": "error",
     "ename": "NameError",
     "evalue": "name 'extract_onyms' is not defined",
     "traceback": [
      "\u001b[0;31m---------------------------------------------------------------------------\u001b[0m",
      "\u001b[0;31mNameError\u001b[0m                                 Traceback (most recent call last)",
      "\u001b[0;32m/tmp/ipykernel_28234/2476247666.py\u001b[0m in \u001b[0;36m<cell line: 0>\u001b[0;34m()\u001b[0m",
      "\u001b[1;32m     69\u001b[0m \u001b[0;34m\u001b[0m\u001b[0m",
      "\u001b[1;32m     70\u001b[0m \u001b[0mtexts\u001b[0m \u001b[0;34m=\u001b[0m \u001b[0mload_texts\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0mfiles\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m",
      "\u001b[0;32m---> 71\u001b[0;31m \u001b[0mcorpus_onyms\u001b[0m \u001b[0;34m=\u001b[0m \u001b[0;34m{\u001b[0m\u001b[0mnovel\u001b[0m\u001b[0;34m:\u001b[0m \u001b[0mextract_onyms\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0mtxt\u001b[0m\u001b[0;34m)\u001b[0m \u001b[0;32mfor\u001b[0m \u001b[0mnovel\u001b[0m\u001b[0;34m,\u001b[0m \u001b[0mtxt\u001b[0m \u001b[0;32min\u001b[0m \u001b[0mtexts\u001b[0m\u001b[0;34m.\u001b[0m\u001b[0mitems\u001b[0m\u001b[0;34m(\u001b[0m\u001b[0;34m)\u001b[0m\u001b[0;34m}\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m",
      "\u001b[0m\u001b[1;32m     72\u001b[0m \u001b[0;34m\u001b[0m\u001b[0m",
      "\u001b[1;32m     73\u001b[0m \u001b[0;31m# ============================================================\u001b[0m\u001b[0;34m\u001b[0m\u001b[0;34m\u001b[0m\u001b[0m"
     ]
    }
   ],
   "source": [
    "import os\n",
    "import re\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import networkx as nx\n",
    "from pyvis.network import Network\n",
    "from sentence_transformers import SentenceTransformer, util\n",
    "import spacy\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "nlp = spacy.load(\"ru_core_news_sm\")\n",
    "sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')\n",
    "\n",
    "files = {\n",
    "    \"Туманность Андромеды\": \"/content/туманность_андромеды.txt\",\n",
    "    \"Час Быка\": \"/content/час_быка.txt\",\n",
    "    \"Сердце Змеи\": \"/content/сердце_змеи.txt\"\n",
    "}\n",
    "\n",
    "def load_texts(file_dict):\n",
    "    texts = {}\n",
    "    encodings = ['utf-8', 'utf-8-sig', 'windows-1251', 'cp1251']\n",
    "\n",
    "    for name, path in file_dict.items():\n",
    "        if os.path.exists(path):\n",
    "            content = None\n",
    "            for enc in encodings:\n",
    "                try:\n",
    "                    with open(path, 'r', encoding=enc) as f:\n",
    "                        content = f.read()\n",
    "                    print(f\"✅ {name} загружен (кодировка: {enc})\")\n",
    "                    break\n",
    "                except UnicodeDecodeError:\n",
    "                    continue\n",
    "\n",
    "            if content is None:\n",
    "                print(f\"❌ Не удалось прочитать {path}. Проверьте кодировку или сохраните в UTF-8.\")\n",
    "            else:\n",
    "                texts[name] = content\n",
    "        else:\n",
    "            print(f\"⚠️ Файл {path} не найден.\")\n",
    "    return texts\n",
    "\n",
    "def extract_onyms(text, min_len=3):\n",
    "    doc = nlp(text)\n",
    "    onyms = set()\n",
    "    for token in doc:\n",
    "        if token.pos_ in {\"PROPN\", \"NOUN\"} and token.is_title and len(token.text) >= min_len:\n",
    "            onyms.add(token.text)\n",
    "\n",
    "    pattern = r'\\b[А-ЯЁ][а-яё]{1,}(?:\\s+(?:[А-ЯЁ][а-яё]{1,}|И))[а-яё]{0,}\\b'\n",
    "    for m in re.finditer(pattern, text):\n",
    "        cand = m.group().strip()\n",
    "        if not cand.startswith(('И ', 'А ', 'Но ', 'О ', 'В ', 'К ', 'С ', 'У ', 'На ', 'По ', 'Для ')):\n",
    "            onyms.add(cand)\n",
    "\n",
    "    return list(onyms)\n",
    "\n",
    "texts = load_texts(files)\n",
    "corpus_onyms = {novel: extract_onyms(txt) for novel, txt in texts.items()}\n",
    "\n",
    "ETYMOLOGY_DB = {\n",
    "    \"Дар Ветер\":   {\"origin\": \"рус./библ.\", \"internal_form\": \"Дар Бога + Ветер\", \"traits\": [\"мудрость\", \"лидерство\", \"гуманизм\"]},\n",
    "    \"Веда\":        {\"origin\": \"слав./индоевр.\", \"internal_form\": \"ведать = знать\", \"traits\": [\"знание\", \"история\", \"дипломатия\"]},\n",
    "    \"Конг\":        {\"origin\": \"вьетн.\", \"internal_form\": \"побеждать\", \"traits\": [\"сила\", \"решительность\", \"победа\"]},\n",
    "    \"Чара Нанди\":  {\"origin\": \"инд./санскрит\", \"internal_form\": \"река Чара + Нанди (счастливый)\", \"traits\": [\"гармония\", \"искусство\", \"танец\"]},\n",
    "    \"Эрг Ноор\":    {\"origin\": \"греч./сканд.\", \"internal_form\": \"ἔργον (работа) + Nórr\", \"traits\": [\"труд\", \"дисциплина\", \"ответственность\"]},\n",
    "    \"Мвен Мас\":    {\"origin\": \"африк.\", \"internal_form\": \"корни банту/суахили\", \"traits\": [\"космос\", \"наука\", \"риск\"]},\n",
    "    \"Чойо Чагас\":  {\"origin\": \"азиат./дисфония\", \"internal_form\": \"тюрк./монгол. основы\", \"traits\": [\"тирания\", \"хитрость\", \"контроль\"]},\n",
    "    \"Шот Шенк\":    {\"origin\": \"азиат./дисфония\", \"internal_form\": \"шумные согласные [ш],[ч],[г]\", \"traits\": [\"власть\", \"жестокость\", \"иерархия\"]},\n",
    "    \"Темное Пламя\":{\"origin\": \"греч./миф\", \"internal_form\": \"огонь Прометея\", \"traits\": [\"жертвенность\", \"тайное знание\", \"бунт\"]},\n",
    "    \"Тантра\":      {\"origin\": \"санскрит/миф\", \"internal_form\": \"нить (Ариадны), путь\", \"traits\": [\"связь\", \"мудрость\", \"путь\"]},\n",
    "    \"Парус\":       {\"origin\": \"греч./миф\", \"internal_form\": \"чёрный парус Тезея\", \"traits\": [\"гибель\", \"судьба\", \"трагедия\"]},\n",
    "    \"Зирда\":       {\"origin\": \"тюрк./каз.\", \"internal_form\": \"зират = могила\", \"traits\": [\"смерть\", \"экология\", \"ядерная угроза\"]}\n",
    "}\n",
    "\n",
    "CONCEPT_CLUSTERS = {\n",
    "    \"сила\": [\"сила\", \"мощь\", \"власть\", \"победа\", \"энергия\"],\n",
    "    \"знание\": [\"знание\", \"мудрость\", \"наука\", \"истина\", \"опыт\"],\n",
    "    \"тирания\": [\"тирания\", \"рабство\", \"контроль\", \"страх\", \"насилие\", \"иерархия\"],\n",
    "    \"экология\": [\"природа\", \"лес\", \"дерево\", \"экология\", \"жизнь\", \"гибель планеты\"],\n",
    "    \"мифология\": [\"бог\", \"миф\", \"легенда\", \"Прометей\", \"Тезей\", \"Ариадна\", \"нить\", \"огонь\", \"космос\", \"судьба\"]\n",
    "}\n",
    "\n",
    "concept_vectors = {k: sbert.encode(v) for k, v in CONCEPT_CLUSTERS.items()}\n",
    "\n",
    "def analyze_semantics(onym):\n",
    "    vec = sbert.encode([onym])[0]\n",
    "    scores = {k: float(util.cos_sim(vec, vecs).mean()) for k, vecs in concept_vectors.items()}\n",
    "    top2 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:2]\n",
    "    return {k: round(v, 3) for k, v in top2 if v > 0.4}\n",
    "\n",
    "def detect_myth_ref(onym, threshold=0.55):\n",
    "    sims = util.cos_sim(sbert.encode([onym]), concept_vectors[\"мифология\"])\n",
    "    return float(sims.mean()) > threshold\n",
    "\n",
    "def run_analysis_pipeline(corpus):\n",
    "    records = []\n",
    "    G = nx.Graph()\n",
    "    for novel, onym_list in corpus.items():\n",
    "        for onym in onym_list:\n",
    "            db_info = ETYMOLOGY_DB.get(onym, {})\n",
    "            origin = db_info.get(\"origin\", \"не определено\")\n",
    "            internal = db_info.get(\"internal_form\", \"\")\n",
    "            traits = list(db_info.get(\"traits\", []))\n",
    "\n",
    "            sem_links = analyze_semantics(onym)\n",
    "            is_myth = detect_myth_ref(onym)\n",
    "            if is_myth and \"мифология\" not in db_info.get(\"traits\", []):\n",
    "                traits.append(\"мифологизация\")\n",
    "\n",
    "            records.append({\n",
    "                \"novel\": novel, \"onym\": onym, \"origin\": origin,\n",
    "                \"internal_form\": internal, \"traits\": \", \".join(set(traits)),\n",
    "                \"semantic_links\": sem_links, \"myth_ref\": is_myth\n",
    "            })\n",
    "\n",
    "            G.add_node(onym, type=\"onym\", novel=novel)\n",
    "            if origin != \"не определено\":\n",
    "                G.add_node(origin, type=\"origin\")\n",
    "                G.add_edge(onym, origin, label=\"происходит из\")\n",
    "            for t in set(traits):\n",
    "                G.add_node(t, type=\"trait\")\n",
    "                G.add_edge(onym, t, label=\"характеризует\")\n",
    "            if is_myth:\n",
    "                G.add_node(\"мифология\", type=\"cluster\")\n",
    "                G.add_edge(onym, \"мифология\", label=\"отсылка к\")\n",
    "            for k, v in sem_links.items():\n",
    "                G.add_node(k, type=\"concept\")\n",
    "                G.add_edge(onym, k, label=f\"близко ({v})\")\n",
    "\n",
    "    df = pd.DataFrame(records)\n",
    "    return df, G\n",
    "\n",
    "df_result, G = run_analysis_pipeline(corpus_onyms)\n",
    "\n",
    "df_result.to_csv(\"efremov_etymology_semantics.csv\", index=False, encoding=\"utf-8-sig\")\n",
    "print(\"✅ Таблица сохранена: efremov_etymology_semantics.csv\")\n",
    "\n",
    "try:\n",
    "    net = Network(height='750px', width='100%', bgcolor='#1a1a2e', font_color='white', directed=False)\n",
    "    for node, attrs in G.nodes(data=True):\n",
    "        net.add_node(str(node), label=str(node), title=str(attrs.get('type', '')))\n",
    "    for u, v, attrs in G.edges(data=True):\n",
    "        net.add_edge(str(u), str(v), label=str(attrs.get('label', '')), title=str(attrs.get('label', '')))\n",
    "    net.toggle_physics(True)\n",
    "    net.write_html(\"efremov_semantic_graph.html\")\n",
    "    print(\"🌐 Интерактивный граф: efremov_semantic_graph.html\")\n",
    "except Exception as e:\n",
    "    print(f\"⚠️ pyvis недоступен ({e}). Создаю PNG-фоллбэк...\")\n",
    "    plt.figure(figsize=(14, 9))\n",
    "    pos = nx.spring_layout(G, k=0.4, seed=42)\n",
    "    nx.draw(G, pos, with_labels=True, node_size=250, font_size=7, node_color='lightblue', edge_color='gray')\n",
    "    plt.title(\"Семантический граф онимов Ефремова\")\n",
    "    plt.savefig(\"efremov_graph_fallback.png\", dpi=200)\n",
    "    plt.show()\n",
    "\n",
    "print(\"\\n📊 Статистика по происхождению основ:\")\n",
    "print(df_result[\"origin\"].value_counts().head(10))\n",
    "print(\"\\n🔍 Онимы с мифологическими отсылками:\")\n",
    "myth_df = df_result[df_result[\"myth_ref\"] == True]\n",
    "if not myth_df.empty:\n",
    "    print(myth_df[[\"onym\", \"novel\", \"internal_form\"]].to_string(index=False))\n",
    "else:\n",
    "    print(\"Точных совпадений не найдено (расширьте ETYMOLOGY_DB или снизьте порог в detect_myth_ref).\")"
   ]
  }
 ],
 "metadata": {
  "colab": {
   "provenance": []
  },
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 0
}